# Calculate L to help estimate Nref for each model

L~Total_seq_length*(nSNPs_final/nSNPs_total)

Total sequence length is the filtered dataset -- all samples.

HOWEVER we have to deal with projection. Since we project each of the two species down to different values, the only site we cound are those that have at least the number of *chromosomes* (not indivs) represented as the projection for that species.

For example, if we project cusp down to 34 and roem down to 26, we only use sites where at least 34 cusp chroms are represented AND 26 roem chroms.

Then we find the number of total SNPs that also match this criteria... this will be the denominator.

Theta = 4\*mu\*L\*Nref

In [1]:
import vcf
import random
import pysam
import numpy as np
import pandas as pd

In [2]:
def extract_locations_from_vcf(vcf_file):
    # open
    vcf = pysam.VariantFile(vcf_file)

    # init list
    locations = []

    # iterate over vcf
    for record in vcf:
        # get position
        contig = record.chrom
        position = record.pos

        # save location tuple to list
        locations.append((contig, position))
    return np.array(locations)

def sample_locations(array, gap_size=50):
    """
    returns list of rad loci separated by gaps larger than min 'gap_size'
    (if a contig has only one locus, sample that single locus)

    :param array: np array of contig names, positions
    :param gap_size: gap size threshold defining separate loci
    :return: list of sampled locations
    """
    #sampled_locations = []
    previous_contig = None
    previous_position = None
    current_locus = []
    loci = []

    for contig, position in array:
        position = int(position)

        # check if we moved to a new contig 
        #or if the gap is larger than the threshold
        if contig != previous_contig or (previous_position is not None and position - previous_position > gap_size):
            if current_locus:
                start = min([i[1] for i in current_locus])
                stop = max([i[1] for i in current_locus])
                loci.append([previous_contig, start, stop])
                
                # samp a location from the current locus and reset it
                #sampled_locations.append(random.choice(current_locus))
                current_locus = []

        # Update the current locus and previous position
        current_locus.append((contig, position))
        previous_contig, previous_position = contig, position

    # samp from the last locus if it's not empty
    if current_locus:
        #sampled_locations.append(random.choice(current_locus))
        start = min([i[1] for i in current_locus])
        stop = max([i[1] for i in current_locus])
        loci.append([previous_contig, start, stop])

    return loci

In [3]:
all_vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned2/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/populations.all.lFilt.iFilt.vcf.gz"
snps_vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned2/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/populations.snps.lFilt.iFilt.vcf.gz"
locations = extract_locations_from_vcf(all_vcf_path)
loci = sample_locations(locations)
vcf_ = pysam.VariantFile(all_vcf_path)

[W::hts_idx_load3] The index file is older than the data file: /n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned2/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/populations.all.lFilt.iFilt.vcf.gz.csi
[W::hts_idx_load3] The index file is older than the data file: /n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned2/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/populations.all.lFilt.iFilt.vcf.gz.csi


In [4]:
len(loci)

62981

# Load in individuals matched to populations

In [7]:
ind2pop = pd.read_csv('./ind2pop_dadi.txt',sep='\t',header=None)

In [8]:
ind2pop

,0,1
0,656A-10,sym
1,656A-23-1,sym
2,656A-26-1,sym
3,656A-29-1,sym
4,656A-4,sym
...,...,...
300,831-21,sym
301,831-22,sym
302,831-3,sym
303,831-5,sym


# Get the total number of sites for sym and allo

In [11]:
sp1 = 'sym'
sp1_proj = 80 # put projection value here
sp1samps = set(ind2pop[ind2pop[1].eq(sp1)][0])

sp2 = 'allo'
sp2_proj = 70 # put projection value here
sp2samps = set(ind2pop[ind2pop[1].eq(sp2)][0])

num_sites = 0

counter = 0
for locus in loci:
    loc_iter = vcf_.fetch(locus[0], locus[1], locus[2])
    
    for variant in loc_iter: # for each variant
        # fast version!
        num_samps_sp1 = np.sum(np.sum(~pd.DataFrame([variant.samples[i].alleles for i in sp1samps]).isnull(),axis=1)>0)
        num_samps_sp2 = np.sum(np.sum(~pd.DataFrame([variant.samples[i].alleles for i in sp2samps]).isnull(),axis=1)>0)

        if num_samps_sp1 >= sp1_proj and num_samps_sp2 >= sp2_proj:
            num_sites += 1
    if not counter % 500:
        print(counter)
    counter += 1

0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
8500
9000
9500
10000
10500
11000
11500
12000
12500
13000
13500
14000
14500
15000
15500
16000
16500
17000
17500
18000
18500
19000
19500
20000
20500
21000
21500
22000
22500
23000
23500
24000
24500
25000
25500
26000
26500
27000
27500
28000
28500
29000
29500
30000
30500
31000
31500
32000
32500
33000
33500
34000
34500
35000
35500
36000
36500
37000
37500
38000
38500
39000
39500
40000
40500
41000
41500
42000
42500
43000
43500
44000
44500
45000
45500
46000
46500
47000
47500
48000
48500
49000
49500
50000
50500
51000
51500
52000
52500
53000
53500
54000
54500
55000
55500
56000
56500
57000
57500
58000
58500
59000
59500
60000
60500
61000
61500
62000
62500


In [12]:
num_sites

2706808

# SNPs

In [5]:
#snps_vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/populations.snps.lFilt.iFilt.vcf.gz"
vcf_ = pysam.VariantFile(snps_vcf_path)
#ind2pop = pd.read_csv('./dadi_ddrad/ind2pop_dadi.txt',sep='\t',header=None)

[W::hts_idx_load3] The index file is older than the data file: /n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned2/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/populations.snps.lFilt.iFilt.vcf.gz.csi


### sym & allo

In [9]:
sp1 = 'sym'
sp1_proj = 80 # put projection value here
sp1samps = set(ind2pop[ind2pop[1].eq(sp1)][0])

sp2 = 'allo'
sp2_proj = 70 # put projection value here
sp2samps = set(ind2pop[ind2pop[1].eq(sp2)][0])

num_sites = 0

counter = 0
for locus in loci:
    loc_iter = vcf_.fetch(locus[0], locus[1], locus[2])
    
    for variant in loc_iter: # for each variant
        # fast version!
        num_samps_sp1 = np.sum(np.sum(~pd.DataFrame([variant.samples[i].alleles for i in sp1samps]).isnull(),axis=1)>0)
        num_samps_sp2 = np.sum(np.sum(~pd.DataFrame([variant.samples[i].alleles for i in sp2samps]).isnull(),axis=1)>0)

        if num_samps_sp1 >= sp1_proj and num_samps_sp2 >= sp2_proj:
            num_sites += 1
    if not counter % 500:
        print(counter)
    counter += 1

0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
8500
9000
9500
10000
10500
11000
11500
12000
12500
13000
13500
14000
14500
15000
15500
16000
16500
17000
17500
18000
18500
19000
19500
20000
20500
21000
21500
22000
22500
23000
23500
24000
24500
25000
25500
26000
26500
27000
27500
28000
28500
29000
29500
30000
30500
31000
31500
32000
32500
33000
33500
34000
34500
35000
35500
36000
36500
37000
37500
38000
38500
39000
39500
40000
40500
41000
41500
42000
42500
43000
43500
44000
44500
45000
45500
46000
46500
47000
47500
48000
48500
49000
49500
50000
50500
51000
51500
52000
52500
53000
53500
54000
54500
55000
55500
56000
56500
57000
57500
58000
58500
59000
59500
60000
60500
61000
61500
62000
62500


In [10]:
num_sites

575010

# Taken from above:

L for sym and allo is: (3944 / 575010) * 2706808 = **18566.0262465**  
* 3944 segregating sites after projection calculated by dadi
* 575010 total SNPs that could be included after projection
* 2706808 total sites that could be included after projection

In [12]:
304.94 / 4 / 1e-8 / 18566.03

410615.5166182539